In [1]:
!pip install pymupdf
import fitz
import os
import torch 
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

/home/ashant/Desktop/75dayplan/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_pdf(path):
    doc=fitz.open(path)
    full_text=""
    for page in doc:
        full_text+=page.get_text()
    return full_text

pdf_path ="/home/ashant/Desktop/75dayplan/projects/rag/attention.pdf"
document=load_pdf(pdf_path)
print(document[:1000])

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Exper

In [3]:
def smart_chunk(text,chunk_size=1000,overlap=100):
    sentences =text.split(". ")
    chunks =[]
    current_chunk=""
    for sentence in sentences:
        if len(current_chunk)+len(sentence) <chunk_size:
            current_chunk +=sentence +". "
        else:
            chunks.append(current_chunk.strip())
            current_chunk=sentence +". "
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

chunks= smart_chunk(document)
# for i,chunk in enumerate(chunks):
#     print(f"\nChunk {i}: \n{chunk}")

print(chunks[1])


Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring significantly
less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-
to-German translation task, improving over the existing best results, including
ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task,
our model establishes a new single-model state-of-the-art BLEU score of 41.8 after
training for 3.5 days on eight GPUs, a small fraction of the training costs of the
best models from the literature. We show that the Transformer generalizes well to
other tasks by applying it successfully to English constituency parsing both with
large and limited training data.
∗Equal contribution. Listing order is random. Jakob proposed replacing RNNs with self-attention and started
the effort to evaluate this idea.


In [5]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

chunks =smart_chunk(document)

chunk_embeddings = embedder.encode(
    chunks,convert_to_numpy=True,normalize_embeddings=True)
dimension = chunk_embeddings.shape[1]
index=faiss.IndexFlatIP(dimension)
index.add(chunk_embeddings)

print("Indexed chunks:",index.ntotal)

Indexed chunks: 46


In [6]:
faiss.write_index(index,"rag_index.faiss")
np.save("chunks.npy",np.array(chunks))

In [7]:
index = faiss.read_index("rag_index.faiss")
chunks = np.load("chunks.npy", allow_pickle=True)


In [8]:
def retrieve(query, k=3):
    query_embedding = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    scores, indices = index.search(query_embedding, k)
    
    results = [chunks[i] for i in indices[0]]
    
    return results, scores


In [9]:
query = "who are the author of Attention is all you need to know?"
results,scores=retrieve(query)

for i,r in enumerate(results):
    print(f"\nRank {i+1} | score: {scores[0][i]:.4f}")
    print(r)


Rank 1 | score: 0.4103
ACL, August 2013.
12
Attention Visualizations
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS>
<pad>
<pad>
<pad>
<pad>
<pad>
<pad>
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS>
<pad>
<pad>
<pad>
<pad>
<pad>
<pad>
Figure 3: An example of the attention mechanism following long-distance dependencies in the
encoder self-attention in layer 5 of 6. Many of the attention heads attend to a distant dependency of
the verb ‘making’, completing the phrase ‘making...more difficult’. Attentions here shown only for
the word ‘making’. Different colors represent different heads.

Rank 2 | score: 0.3829
Even with k = n, however, the complexity of a separable
convolution is equal to the combination of a self-attention layer and a point-wise feed-forward layer,
the

In [10]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


In [11]:
def rerank(query, retrieved_chunks):
    pairs = [(query, chunk) for chunk in retrieved_chunks]
    scores = reranker.predict(pairs)
    
    ranked = sorted(
        zip(retrieved_chunks, scores),
        key=lambda x: x[1],
        reverse=True
    )
    
    return ranked
reranked = rerank(query, results)

for chunk, score in reranked:
    print("\nRerank Score:", score)
    print(chunk)



Rerank Score: -9.462435
ACL, August 2013.
12
Attention Visualizations
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS>
<pad>
<pad>
<pad>
<pad>
<pad>
<pad>
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS>
<pad>
<pad>
<pad>
<pad>
<pad>
<pad>
Figure 3: An example of the attention mechanism following long-distance dependencies in the
encoder self-attention in layer 5 of 6. Many of the attention heads attend to a distant dependency of
the verb ‘making’, completing the phrase ‘making...more difficult’. Attentions here shown only for
the word ‘making’. Different colors represent different heads.

Rerank Score: -9.868139
Note that the attentions are very sharp for this word.
14
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in


In [12]:
def build_prompt(query, retrieved_chunks):
    context = "\n\n".join(retrieved_chunks)
    
    prompt = f"""
You are a strict QA system.

Answer ONLY using the context below.
If answer is not present, say "Not found in document."

Context:
{context}

Question:
{query}
"""
    return prompt
print(build_prompt(query,results))


You are a strict QA system.

Answer ONLY using the context below.
If answer is not present, say "Not found in document."

Context:
ACL, August 2013.
12
Attention Visualizations
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS>
<pad>
<pad>
<pad>
<pad>
<pad>
<pad>
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS>
<pad>
<pad>
<pad>
<pad>
<pad>
<pad>
Figure 3: An example of the attention mechanism following long-distance dependencies in the
encoder self-attention in layer 5 of 6. Many of the attention heads attend to a distant dependency of
the verb ‘making’, completing the phrase ‘making...more difficult’. Attentions here shown only for
the word ‘making’. Different colors represent different heads.

Even with k = n, however, the complexity of a separable
convolution is equal 

In [13]:
!pip install transformers accelerate


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "microsoft/phi-2"   # Lightweight

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)


Loading checkpoint shards: 100%|██████████| 2/2 [00:21<00:00, 10.64s/it]
Some parameters are on the meta device because they were offloaded to the cpu.


In [ ]:
def rag_answer(query, k=5):
    
    # Step 1 — Retrieve
    retrieved_chunks, _ = retrieve(query, k=k)
    
    # Step 2 — Rerank
    reranked = rerank(query, retrieved_chunks)
    top_chunks = [chunk for chunk, score in reranked[:3]]
    
    # Step 3 — Build Prompt
    context = "\n\n".join(top_chunks)
    
    prompt = f"""
You are a strict QA system.

Answer ONLY using the context below.
If answer is not present, say "Not found in document."

Context:
{context}

Question:
{query}

Answer:
"""
    
    # Step 4 — Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Step 5 — Generate
    outputs = model.generate(
        **inputs,
        max_new_tokens=20,
        temperature=0.2,
        do_sample=False
    )
    
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return answer

print(rag_answer(query))
